# Cambrian-P: ask questions about your own video

Runs [Cambrian-P](https://github.com/cambrian-mllm/cambrian-p) on a video stored in your Google Drive, with one model or all five released models, and puts their answers side by side.

**Before you start:** Runtime → Change runtime type → **A100 GPU**. Each 7B model needs about 15 GB for its weights alone, so the free T4 is too small.

**How to use**
1. Fill in the settings cell and run it.
2. Run the cells in order. Skip cell 4 (the Drive backup) unless cell 3 just did a fresh install.
3. First time: about 15 minutes of setup. Each model then takes a few minutes to download (about 16 GB each) and a few minutes to load and answer. All five models take very roughly 30–60 minutes.

Answers are saved to Google Drive (`RESULTS_DIR`) as each one is produced.

**Check the load worked:** each model's log should contain a `token_num:` line, `[cam-load] ✓ Loaded model.rec_head.camera_tokens`, and `SUCCESS: All weights in new_state_dict match the loaded model.` Without them, the camera tokens are not being used.

In [ ]:
# @title 1. Settings
VIDEO_PATH = "/content/drive/MyDrive/Cambrian-P/Video_files/IMG_1063.MOV"  # @param {type:"string"}
RUN_ALL_MODELS = True  # @param {type:"boolean"}
MODEL = "Cambrian-P-7B"  # @param ["Cambrian-P-7B", "Cambrian-P-7B-32f", "Cambrian-P-7B-Mix-MA", "Cambrian-P-7B-Mix-3R", "Cambrian-P-7B-Mix-CamS"]
NUM_FRAMES = 0  # @param {type:"integer"}
DELETE_MODELS_AFTER_RUN = False  # @param {type:"boolean"}
RESULTS_DIR = "/content/drive/MyDrive/Cambrian-P/results"  # @param {type:"string"}
BACKUP_PATH = "/content/drive/MyDrive/cambrianp_env.tar"  # @param {type:"string"}
BACKUP_INCLUDES_MODEL = False  # @param {type:"boolean"}
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"  # @param {type:"string"}

# RUN_ALL_MODELS runs every model below in turn; otherwise only MODEL runs.
# NUM_FRAMES = 0 uses the frame count each model was evaluated with (doc/evaluation.md).
DEFAULT_FRAMES = {
    "Cambrian-P-7B": 128,
    "Cambrian-P-7B-32f": 32,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-Mix-3R": 32,
    "Cambrian-P-7B-Mix-CamS": 128,
}
MODELS = list(DEFAULT_FRAMES) if RUN_ALL_MODELS else [MODEL]

# One question per entry. Every model answers all of them.
QUESTIONS = [
    "How many chairs are in the room?",
    "Describe the path the camera takes through the scene.",
]

import json, os
os.environ.update({
    "BACKUP_PATH": BACKUP_PATH, "BACKUP_INCLUDES_MODEL": str(int(BACKUP_INCLUDES_MODEL)),
    "REPO_URL": REPO_URL,
    # Keep uv's Python inside /content so the Drive backup contains everything the venv needs.
    "UV_PYTHON_INSTALL_DIR": "/content/python",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
})
with open("/content/questions.json", "w") as f:
    json.dump(QUESTIONS, f)
for name in MODELS:
    print(f"{name}: {NUM_FRAMES or DEFAULT_FRAMES[name]} frames")
print(f"{len(QUESTIONS)} question(s) on {VIDEO_PATH}")

In [ ]:
# @title 2. Check the GPU and mount Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Runtime → Change runtime type → A100 GPU, then run again.")
print("GPU:", gpu)
if int(gpu.split(",")[-1].split()[0]) < 30000:
    print("Warning: under 30 GB of GPU memory. If you run out of memory, set NUM_FRAMES to 32 in cell 1.")

from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists(VIDEO_PATH):
    print(f"Warning: video not found at {VIDEO_PATH}. Check the path in cell 1 (browse it in the Files panel on the left).")

In [ ]:
%%bash
# @title 3. Set up the environment (skips itself if already done)
set -e
cd /content
if /content/venv/bin/python -c "import torch" 2>/dev/null && [ -d /content/cambrian-p ]; then
    echo "Environment already set up in this session."
elif [ -f "$BACKUP_PATH" ]; then
    echo "Restoring environment from $BACKUP_PATH ..."
    tar xf "$BACKUP_PATH" -C /content
    echo "Restored."
else
    echo "Fresh install (about 10-15 minutes) ..."
    [ -d cambrian-p ] || git clone -q "$REPO_URL" cambrian-p
    pip install -q uv
    # The repo is pinned to Python 3.11, which may differ from Colab's own Python.
    uv venv --seed --python 3.11 /content/venv
    source /content/venv/bin/activate
    cd cambrian-p
    # Same steps and order as the README's "Environment Preparation".
    pip install -q -e ./vggt
    pip install -q hydra-core tensorboard iopath wcmatch fvcore
    pip install -q --upgrade pip
    pip install -q -e ".[train]"
    pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
        --index-url https://download.pytorch.org/whl/cu121
    pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
    # Used by lmms-eval's model wrapper. flash-attn is skipped: the wrapper always uses PyTorch's
    # built-in attention (sdpa) on torch >= 2.1.2, so it would never be used.
    pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
    echo "Install finished."
fi
# pyarrow: newer releases need NumPy 2, but the repo pins NumPy 1.26.1. Also fixes backups made before this pin.
/content/venv/bin/pip install -q pyarrow==15.0.2

In [ ]:
%%bash
# @title 4. (Optional, run once after a fresh install) Save the environment to Drive for faster future sessions
# Without the model the archive is several GB; with it, add about 15 GB (needs paid Drive storage).
set -e
cd /content
EXCLUDE="--exclude=cambrian-p/ckpts"
[ "$BACKUP_INCLUDES_MODEL" = "1" ] && EXCLUDE=""
for d in python venv cambrian-p; do
    if [ ! -d "$d" ]; then
        echo "Nothing to save: /content/$d is missing. Run cell 3 first; your existing backup was left untouched."
        exit 1
    fi
done
# Write to a temporary file first so a failed or interrupted save can't damage an existing backup.
echo "Writing $BACKUP_PATH ..."
tar cf "$BACKUP_PATH.partial" $EXCLUDE python venv cambrian-p
mv "$BACKUP_PATH.partial" "$BACKUP_PATH"
ls -lh "$BACKUP_PATH"

In [ ]:
%%writefile /content/run_video.py
"""Answer questions about one video with one Cambrian-P model, reusing the lmms-eval model wrapper."""
import argparse
import json
import os
import time

from PIL import Image
import cambrianp.utils as cambrianp_utils
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from cambrianp.conversation import conv_templates
from cambrianp.mm_utils import tokenizer_image_token

ap = argparse.ArgumentParser()
ap.add_argument("--model_path", required=True)
ap.add_argument("--video", required=True)
ap.add_argument("--questions_file", required=True)
ap.add_argument("--num_frames", type=int, default=128)
ap.add_argument("--output_json", required=True)
args = ap.parse_args()

with open(args.questions_file) as f:
    questions = json.load(f)

# For any checkpoint path containing "cambrian", the loader converts the config to LLaVA-OneVision
# style (cambrianp/utils.py). That conversion is meant for starting training from Cambrian-S. Released
# Cambrian-P checkpoints were saved already converted, so converting again overwrites vocab_size
# (152064 vs the weights' 151672) and deletes the camera-head settings, which silently drops the
# per-frame camera tokens. Keep the checkpoint's config as saved, and switch the camera head on
# (use_camera_tokens only takes effect with load_rec_model, llava_arch.py:556; the pose eval sets it too).
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        # Path to the authors' VGGT init file; the trained head weights come from the checkpoint itself.
        config.rec_model_path = "None"
    return config

cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config

start = time.time()

# Flags from doc/evaluation.md.
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")

# Same frame preprocessing as CambrianP.generate_until for video inputs.
frames = m._load_video_frames(args.video, m.max_frames_num)
image_size = list(frames.shape[1:3])
bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in frames]
pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()

pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id
result = {"model": os.path.basename(args.model_path.rstrip("/")), "video": args.video,
          "num_frames": args.num_frames, "load_seconds": round(time.time() - start, 1), "answers": []}
for question in questions:
    t = time.time()
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    answer = m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", {}, args.video)[0]
    print(f"\nQ: {question}\nA: {answer}", flush=True)
    result["answers"].append({"question": question, "answer": answer, "seconds": round(time.time() - t, 1)})
    # Rewrite after every answer so a crash part-way still keeps what was answered.
    with open(args.output_json, "w") as f:
        json.dump(result, f, indent=1)

In [ ]:
# @title 5. Run the model(s) on your video
import shutil, subprocess, time
from datetime import datetime

VENV_BIN = "/content/venv/bin"
REPO = "/content/cambrian-p"
env = dict(os.environ, PYTHONPATH=f"{REPO}/lmms-eval:{REPO}", PATH=f"{VENV_BIN}:{os.environ['PATH']}")

# Check the video opens before spending time on models.
probe = subprocess.run([f"{VENV_BIN}/python", "-c",
    "import sys; from decord import VideoReader, cpu; vr = VideoReader(sys.argv[1], ctx=cpu(0)); "
    "print(f'{len(vr)} frames, {vr.get_avg_fps():.1f} fps, frame size (H, W) = {vr[0].shape[:2]}')", VIDEO_PATH],
    capture_output=True, text=True, env=env)
if probe.returncode != 0:
    raise SystemExit(f"Could not open the video:\n{probe.stderr[-2000:]}")
print("Video:", probe.stdout.strip())

stamp = datetime.now().strftime("%Y%m%d-%H%M")
video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
run_dir = os.path.join(RESULTS_DIR, f"{video_name}_{stamp}")
os.makedirs(run_dir, exist_ok=True)
print("Saving results to", run_dir)

def run(cmd):
    """Run a command, streaming its output; return (exit code, last lines of output)."""
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-40:]
    return p.wait(), "".join(tail)

results = {}
for name in MODELS:
    frames = NUM_FRAMES or DEFAULT_FRAMES[name]
    print(f"\n{'=' * 30} {name} ({frames} frames) {'=' * 30}")
    t = time.time()
    ckpt = f"{REPO}/ckpts/{name}"
    if not os.path.isfile(f"{ckpt}/config.json"):
        free_gb = shutil.disk_usage("/content").free / 1e9
        if free_gb < 20:
            print(f"Only {free_gb:.0f} GB of disk free; each model needs about 16 GB. "
                  "Tick DELETE_MODELS_AFTER_RUN in cell 1 and rerun cell 1 and this cell.")
            results[name] = {"error": "not enough disk space to download"}
            continue
        print(f"Downloading nyu-visionx/{name} (about 16 GB) ...")
        code_, tail = run([f"{VENV_BIN}/huggingface-cli", "download", f"nyu-visionx/{name}", "--local-dir", ckpt])
        if code_ != 0:
            results[name] = {"error": "download failed", "log": tail}
            continue
    out_json = os.path.join(run_dir, f"{name}.json")
    code_, tail = run([f"{VENV_BIN}/python", "/content/run_video.py", "--model_path", f"ckpts/{name}",
                       "--video", VIDEO_PATH, "--questions_file", "/content/questions.json",
                       "--num_frames", str(frames), "--output_json", out_json])
    results[name] = json.load(open(out_json)) if os.path.exists(out_json) else {}
    if code_ != 0:
        results[name].update(error=f"run failed (exit code {code_})", log=tail)
    if DELETE_MODELS_AFTER_RUN:
        shutil.rmtree(ckpt, ignore_errors=True)
    print(f"[{name}] finished in {(time.time() - t) / 60:.1f} min" + (" with an error" if code_ else ""))

with open(os.path.join(run_dir, "all_results.json"), "w") as f:
    json.dump({"video": VIDEO_PATH, "questions": QUESTIONS, "results": results}, f, indent=1)
print("\nDone. Results saved in", run_dir)

In [ ]:
# @title 6. Compare the answers
from IPython.display import Markdown, display

lines = [f"# Cambrian-P answers for `{os.path.basename(VIDEO_PATH)}`", ""]
for i, q in enumerate(QUESTIONS):
    lines += [f"## {q}", "", "| Model | Frames | Answer |", "|---|---|---|"]
    for name in MODELS:
        r = results.get(name, {})
        answers = r.get("answers", [])
        a = answers[i]["answer"] if i < len(answers) else f"*{r.get('error', 'no answer')}*"
        a = a.replace("|", "\\|").replace("\n", " ")
        lines.append(f"| {name} | {r.get('num_frames', NUM_FRAMES or DEFAULT_FRAMES[name])} | {a} |")
    lines.append("")
failed = {n: r for n, r in results.items() if "error" in r}
for name, r in failed.items():
    lines += [f"**{name} failed:** {r['error']}. The end of its log is in `all_results.json`.", ""]
report = "\n".join(lines)
with open(os.path.join(run_dir, "comparison.md"), "w") as f:
    f.write(report)
display(Markdown(report))
print("Also saved to", os.path.join(run_dir, "comparison.md"))